In [1]:
import numpy as np
import pandas as pd
import pulp
import json
from pathlib import Path
import torch
import os
import importlib
import sensitivity
importlib.reload(sensitivity)
from sensitivity import heuristic

In [2]:
import pandas as pd

df = pd.read_csv("../model_summary.csv")

# Extract architecture number from filename
df["arch"] = df["file"].str.extract(r"arch(\d+)").astype(int)

# Compute mean RMSE for each architecture
mean_rmse = (
    df.groupby("arch", as_index=False)["rmse"]
      .mean()
      .rename(columns={"rmse": "mean_rmse"})
      .sort_values("arch")
)

display(mean_rmse)


,arch,mean_rmse
0,2,0.048523
1,3,0.028985
2,4,0.028340
3,5,0.027522


In [3]:
tradeoff_map = np.array([
    # left_rmse   track_rmse   heading_rmse   params
    [0.03528404, 0.06961147, 0.04067444,  21997],   # Arch 2
    [0.01816354, 0.02976013, 0.03903201,  34405],   # Arch 3
    [0.01831261, 0.03011044, 0.03659838,  69105],   # Arch 4
    [0.01710328, 0.03466046, 0.03374820, 120235],   # Arch 5
])

In [4]:
df = pd.read_csv("../model_summary.csv")

# Extract architecture index from filename (arch2, arch3, ...)
df["arch"] = df["file"].str.extract(r"arch(\d+)").astype(int)

# Pivot RMSE metrics into columns
rmse_pivot = df.pivot_table(
    index="arch",
    columns="target",
    values="rmse"
)

# Rename columns to match desired order
rmse_pivot = rmse_pivot.rename(columns={
    "left_wall_dist": "left_rmse",
    "track_width": "track_rmse",
    "heading_error": "heading_rmse"
})

# Ensure correct column order
rmse_pivot = rmse_pivot[["left_rmse", "track_rmse", "heading_rmse"]]

# Aggregate params per architecture (they are repeated in CSV)
params = df.groupby("arch")["params"].first()

# Combine everything
tradeoff_df = rmse_pivot.copy()
tradeoff_df["params"] = params

# Sort by architecture index
tradeoff_df = tradeoff_df.sort_index()

# Convert to numpy array
tradeoff_map = tradeoff_df.to_numpy()

print(tradeoff_map)

[[3.52840386e-02 6.96114674e-02 4.06744368e-02 2.19970000e+04]
 [1.81635413e-02 2.97601316e-02 3.90320122e-02 3.44050000e+04]
 [1.83126107e-02 3.01104449e-02 3.65983769e-02 6.91050000e+04]
 [1.70793664e-02 2.86457464e-02 3.68414186e-02 1.20235000e+05]]


In [5]:
sensitivity = [5.829618417354863, 5.829618417354863, 6.783432564382072]

In [12]:
selection = heuristic(
    sensitivity=sensitivity,
    budget=350000,
    tradeoff_map=tradeoff_map,
)
print(selection)

[2, 3, 3]


In [15]:
jsonl_path = Path("./output-dir/b481df_2/metrics.jsonl")

results = []

with open(jsonl_path, "r") as f:
    for line in f:
        row = json.loads(line)

        if row["map"] == "F1/Shanghai/Shanghai":
            continue

        runs = row["runs"]

        baseline_rmse = None
        arch8_rmse = None

        for run in runs:
            if run["label"] == "comb1":
                baseline_rmse = run["rmse"]
            else:
                # assumes the other entry is your arch8 run
                arch8_rmse = run["rmse"]

        results.append({
            "map": row["map"],
            "baseline_rmse": baseline_rmse,
            "arch8_rmse": arch8_rmse,
        })


df = pd.DataFrame(results)

        

df.loc[len(df)] = {
    "map": "MEAN",
    "baseline_rmse": df["baseline_rmse"].mean(),
    "arch8_rmse": df["arch8_rmse"].mean(),
            }

df

,map,baseline_rmse,arch8_rmse
0,F1/MexicoCity/MexicoCity,0.171700,0.156500
1,F1/Monza/Monza,0.106900,0.104400
2,F1/Nuerburgring/Nuerburgring,0.170500,0.134600
3,F1/Silverstone/Silverstone,0.142800,0.142600
4,F1/Sochi/Sochi,0.138100,0.124800
5,F1/Spa/Spa,0.131500,0.124200
6,MEAN,0.143583,0.131183


In [13]:

# heuristic3 returns level indices, convert to arch numbers
track_arch = selection[0] + 1
left_arch = selection[1] + 1
heading_arch = selection[2] + 1

# --------------------------------------------------
# Arch1-7 parameter lookup table
# --------------------------------------------------

arch_sizes = {
    1: 21997,
    2: 34405,
    3: 69105,
    4: 120235,
}

baseline_sizes = {
    "track_width": arch_sizes[track_arch],
    "left_wall_dist": arch_sizes[left_arch],
    "heading_error": arch_sizes[heading_arch],
}

# --------------------------------------------------
# Arch8 models
# --------------------------------------------------

track_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/b481df/track_width_arch8_trial297.pt"

left_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/b481df/left_wall_dist_arch8_trial297.pt"

heading_path = "/home/tingan/NAS-Sensitivity/safety-nas/test-best-runs-tp0/b481df/heading_error_arch8_trial297.pt"

def count_params(path):
    model = torch.jit.load(path, map_location="cpu")
    return sum(p.numel() for p in model.parameters())

arch8_sizes = {
    "track_width": count_params(track_path),
    "left_wall_dist": count_params(left_path),
    "heading_error": count_params(heading_path),
}

# --------------------------------------------------
# Build comparison table
# --------------------------------------------------

df = pd.DataFrame({
    "target": ["track_width", "left_wall_dist", "heading_error"],
    "baseline_arch": [track_arch, left_arch, heading_arch],
    "baseline_params": [
        baseline_sizes["track_width"],
        baseline_sizes["left_wall_dist"],
        baseline_sizes["heading_error"],
    ],
    "arch8_params": [
        arch8_sizes["track_width"],
        arch8_sizes["left_wall_dist"],
        arch8_sizes["heading_error"],
    ],
})

df["param_difference"] = (
    df["arch8_params"] - df["baseline_params"]
)

# Total row
df.loc[len(df)] = {
    "target": "TOTAL",
    "baseline_arch": "",
    "baseline_params": df["baseline_params"].sum(),
    "arch8_params": df["arch8_params"].sum(),
    "param_difference":
        df["arch8_params"].sum()
        - df["baseline_params"].sum(),
}

df

,target,baseline_arch,baseline_params,arch8_params,param_difference
0,track_width,3,69105,34693,-34412
1,left_wall_dist,4,120235,69057,-51178
2,heading_error,4,120235,69009,-51226
3,TOTAL,,309575,172759,-136816
